# Import libraries & data

In [ ]:
import gzip
import pandas as pd
import json
import os
import warnings
import seaborn as sns
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
path = '/content/drive/MyDrive/Dataset for Python/'

In [ ]:
payment_report = pd.read_csv(path+"payment_report.csv", encoding = 'utf-8')
product = pd.read_csv(path+"product.csv", encoding = 'utf-8')
transactions = pd.read_csv(path+"transactions.csv", encoding = 'utf-8')

In [ ]:
transactions.head(2)

In [ ]:
product.head(2)

In [ ]:
payment_report.head(2)

In [ ]:
df_payment_enriched = payment_report.merge(product, on='product_id', how='left')
df_payment_enriched.head()

# Part 1: EDA

## EDA payment_report.csv & product.csv
- Missing data: 2% (22 rows) of `category, team_own` because some Product IDs in `payment_report` df don't exist in `product` df -> Next step: No action
- Duplicates: 0 row -> Next step: No action
- Incorrect data types: report_month -> Next step: change data type
- Incorrect values: 0 row -> Next step: No action

In [ ]:
#kiểm tra data có bao nhiêu dòng, bao nhiêu cột
print(df_payment_enriched.shape)
#kiểm tra thông tin dtype của data
df_payment_enriched.info()

In [ ]:
df_payment_enriched # Sai data type report_month
df_payment_enriched["report_month"]= pd.to_datetime(df_payment_enriched["report_month"])
df_payment_enriched

In [ ]:
df_payment_enriched.dtypes

In [ ]:
!pip install ydata-profiling
from ydata_profiling import ProfileReport

In [ ]:
profile = ProfileReport(df_payment_enriched, explorative=True)

In [ ]:
profile.to_notebook_iframe()

## EDA transactions.csv
- Missing data: `sender_id, receiver_id` -> Next step: No action
- Duplicates: 28 rows -> Next step: Delete rows
- Incorrect data types: 0 row -> Next step: No action
- Incorrect values: 0 row -> Next step: No action

In [ ]:
transactions.info()

In [ ]:
profile = ProfileReport(transactions, explorative=True)

In [ ]:
profile.to_notebook_iframe()

In [ ]:
transactions = transactions.drop_duplicates()
transactions

# Part II: Data Wrangling

## 1. Top 3 product_ids with the highest volume.

In [ ]:
# lấy 3 product_id có volume cao nhất
payment_enriched_cleaned1 = df_payment_enriched
sum_product= payment_enriched_cleaned1.groupby(by = 'product_id', as_index = False)["volume"].sum().sort_values(by = 'volume',ascending= False)
sum_product.head(3)

## 2. Given that 1 product_id is only owed by 1 team, are there any abnormal products against this rule?

In [ ]:
abnormal_products=  payment_enriched_cleaned1.groupby("product_id")["team_own"].nunique()
abnormal_products[abnormal_products != 1]
#ko có team nào cùng sở hửu 1 product

## 3. Find the team has had the lowest performance (lowest volume) since Q2.2023. Find the category that contributes the least to that team.

In [ ]:
# category that contributes the least to that Lowest team  since Q2.2023
payment_report_merge_from_Q2_2023=payment_enriched_cleaned1[payment_enriched_cleaned1['report_month']>="2023-04-01"]
lowest_team = payment_report_merge_from_Q2_2023.groupby("team_own")["volume"].sum()
lowest_team=lowest_team.sort_values()
lowest_team.head(1)
#APS là team thấp

In [ ]:
team_APS= payment_enriched_cleaned1[(payment_enriched_cleaned1["team_own"]=="APS")&(payment_enriched_cleaned1['report_month']>="2023-04-01")]
Lowst_category_team_APS= team_APS.groupby("category")["volume"].sum()
Lowst_category_team_APS= Lowst_category_team_APS.sort_values()
Lowst_category_team_APS.head(1)
#PXXXXXS đóng góp thấp nhất

## 4. Find the contribution of source_ids of refund transactions (payment_group = ‘refund’), what is the source_id with the highest contribution?

In [ ]:
#Find the contribution of source_ids of refund transactions (payment_group = ‘refund’), what is the source_id with the highest contribution?
refund= payment_enriched_cleaned1[payment_enriched_cleaned1["payment_group"]== "refund"].groupby("source_id")["volume"].sum().sort_values(ascending= False).head(1)
refund

## 5. Define type of transactions (‘transaction_type’) for each row, given:
- transType = 2 & merchant_id = 1205: Bank Transfer Transaction
- transType = 2 & merchant_id = 2260: Withdraw Money Transaction
- transType = 2 & merchant_id = 2270: Top Up Money Transaction
- transType = 2 & others merchant_id: Payment Transaction
- transType = 8, merchant_id = 2250: Transfer Money
Transaction
- transType = 8 & others merchant_id: Split Bill Transaction
- Remained cases are invalid transactions

In [ ]:
def classify_trac(row):
  t_type = row['transType']
  m_id = row['merchant_id']
  if t_type == 2 :
    if m_id == 1205 :
      return "Bank Transfer Transaction"
    elif m_id == 2260:
      return " Withdraw Money Transaction"
    elif m_id == 2270:
      return "Top Up Money Transaction"
    else : return "Payment Transaction"
  elif t_type == 8:
    if m_id == 2250:
      return "Transfer Money Transaction"
    else: return "Split Bill Transaction"
  else: return "Invalid"
transactions["transaction_type"] = transactions.apply(classify_trac,axis=1)


In [ ]:
transactions

## 6. Of each transaction type (excluding invalid transactions): find the number of transactions, volume, senders and receivers

In [ ]:
transactions_new= transactions[transactions["transaction_type"]!="Invalid"]
transactions_new.groupby("transaction_type").agg(num_transactions=('transaction_id', 'count'),total_volume=('volume', 'sum'),unique_senders=('sender_id', 'nunique'),unique_receivers=('receiver_id', 'nunique'))